# Charlotte Project | Week 10 — Class 2
## Pandas Fundamentals: Series, DataFrames, and Production Tables

**Teaching mode:** Interactive Jupyter Notebook · **Duration:** 180 minutes · **Assessment:** Separate Python scripts and GitHub Pull Request

### What you will achieve
1. Explain how Pandas relates to Python and NumPy.
2. Construct and examine `Series` and `DataFrame` objects.
3. Import CSV data and understand columns, indices and data types.
4. Use safe column selection, row selection and basic Boolean filtering.
5. Add calculated columns while retaining missing values.
6. Interpret simple quality checks and prepare for independent scripting.

> The teaching examples use a **small demonstration dataset different from the assessed student CSV**. Do not copy this notebook as your assignment.

**Environment:** VS Code Python + Jupyter extensions, selected Week 10 `.venv`, `pandas`, `numpy`, `ipykernel`. Install with `python -m pip install pandas numpy ipykernel`.

# A — Why Pandas? Introduction to labelled data (25 minutes)

### The problem Pandas solves
Oil-and-gas datasets rarely arrive as a clean numerical matrix with no context. A production table includes well IDs, dates, operational status, field names, and measurements. **Pandas** provides labelled, tabular structures that keep such information together. A **Series** is a one-dimensional labelled collection; a **DataFrame** is a two-dimensional table of Series sharing the same row index.

Pandas builds on NumPy but adds practical features: column names, mixed data types, CSV import/export, missing-value handling, and row labels. It does **not** automatically make bad data trustworthy: analysts must still check types, units and missing values.

**Why it matters in operations:** Without labelled columns, pressure (psi) and temperature (°F) could be confused. With a DataFrame, the code can use `readings["pressure_psi"]` instead of remembering an unnamed array position.

**Key vocabulary:** index = row labels, column = named variable, dtype = type stored for a Series, `NaN` = common missing numeric value.

**Common mistake:** A Series is not a single scalar; it can hold many readings. A DataFrame column is usually a Series.

**Before you run:** Predict whether a Series will display labels, not just values.

In [ ]:
import pandas as pd
import numpy as np
print('Pandas version:', pd.__version__)
print('NumPy version:', np.__version__)
pressure = pd.Series([3100, 2950, 3025], index=['WELL-A', 'WELL-B', 'WELL-C'], name='pressure_psi')
print(pressure)
print('Well B pressure:', pressure.loc['WELL-B'])

### Try it: Series practice (ungraded)
Create a `water_bpd` Series for three wells. Use meaningful labels and select one well with `.loc`. Then describe in one sentence what an index represents.

**Checkpoint:** Explain what `pressure.name`, `pressure.index`, and `pressure.dtype` tell you.

# B — Constructing and inspecting DataFrames (35 minutes)

### Concept introduction
A **DataFrame** resembles a spreadsheet with rows and named columns, but it is manipulated through Python expressions. You can create one from a dictionary of equal-length lists, a list of dictionaries, or a file. Each column may have its own type: strings for well IDs and status, floats for pressure, and timestamps for dates.

Start all unfamiliar datasets with an **inspection routine**: `head()`, `shape`, `columns`, `dtypes`, `info()` and `describe()`. These answer different questions: preview values, dimensions, available fields, inferred types, completeness and numerical range.

**Operational application:** A well may change status across days. Each row should describe one well on one date (a well-day record), rather than one row per well. You must understand the unit of observation before making summaries.

**Common mistakes:** `df.shape` is a property, not `df.shape()`; `df.head()` previews data but does not guarantee every row is valid; inferred numbers can become text if CSV contents are inconsistent.

**Before running:** What do you expect `shape` to return for four rows and seven columns?

In [ ]:
demo = pd.DataFrame({
    'well_id': ['DEMO-A', 'DEMO-A', 'DEMO-B', 'DEMO-C'],
    'date': ['2026-10-01', '2026-10-02', '2026-10-01', '2026-10-01'],
    'status': ['PRODUCING', 'PRODUCING', 'SHUT_IN', 'PRODUCING'],
    'oil_bpd': [800, 820, 0, 640],
    'water_bpd': [180, 185, 0, 210],
    'pressure_psi': [3000, 3020, 2800, 2900],
    'field': ['North', 'North', 'North', 'South']
})
print('Rows and columns:', demo.shape)
print('Columns:', demo.columns.tolist())
print(demo.head())
print(demo.dtypes)

In [ ]:
print('First two rows:')
print(demo.head(2))
print('Numerical summary:')
print(demo.describe())
print('Field summary:')
print(demo['field'].value_counts())

### Mini challenge
How many **unique well IDs** exist? Why is the number of rows larger? Inspect `demo.info()` and identify the numerical columns.

**Important boundary:** Detailed multi-level aggregation (`groupby` with multiple aggregations) is the main focus of Class 3; this lesson introduces only simple counts.

# C — CSV ingestion and reliable data types (30 minutes)

### Concept introduction
Most field reporting systems export tabular observations as CSV. `pd.read_csv()` reads a file into a DataFrame, and `df.to_csv()` writes a DataFrame back to disk. A CSV file stores **text** and has no native date or numeric types; Pandas infers types while reading.

For a `date` column, use `parse_dates=["date"]` when importing or `pd.to_datetime()` after import. Verify with `dtypes`. The resulting values are timestamps rather than ordinary strings, which becomes important for Week 10 time-series classes.

**Missing values:** An empty numeric CSV field normally becomes `NaN`, which represents missing data. Missing production measurements must not automatically be replaced with zero: a shut-in well might truly produce zero, while a missing measurement means the number is unknown. These cases are not interchangeable.

**Common mistakes:** using a path relative to the wrong current working directory; assuming `read_csv` understands every date format; interpreting a missing pressure as zero psi.

**Before running:** We will create a tiny in-memory CSV demonstration. No lab file is loaded here.

In [ ]:
from io import StringIO
demo_csv = StringIO('well_id,date,oil_bpd,pressure_psi\nDEMO-X,2026-10-05,700,3050\nDEMO-Y,2026-10-05,,2990\nDEMO-X,2026-10-06,715,\n')
imported = pd.read_csv(demo_csv, parse_dates=['date'])
print(imported)
print(imported.dtypes)
print('Missing per column:', imported.isna().sum().to_dict())

In [ ]:
print('Date type:', imported['date'].dtype)
print('Missing oil?:', pd.isna(imported.loc[1, 'oil_bpd']))
print('Missing pressure?:', pd.isna(imported.loc[2, 'pressure_psi']))

### Try it
Change one numeric entry in the in-memory CSV to a blank field, rerun, and describe how `isna().sum()` changes. Explain why a missing value is not the same as a zero reading.

# D — Selecting rows and columns safely (35 minutes)

### Concept introduction
An analyst rarely needs every column and row. Use `df["column"]` to select one Series and `df[["column1", "column2"]]` to create a smaller DataFrame. `loc` selects by **label**, while `iloc` selects by **integer position**. Boolean masks select rows satisfying a condition, such as `status == "PRODUCING"`.

**Why it matters:** Operators may want just active production readings for the North field, or just oil and pressure values for a review. Filtering makes operational questions precise.

**Important detail:** With `loc`, the end of a label slice is often included; with `iloc`, the end of a position slice is excluded. For comparisons, use parentheses around each condition and the bitwise operators `&` (and), `|` (or), not Python's `and`/`or`.

**Common mistakes:** `df['oil_bpd', 'status']` is not the usual syntax for selecting two columns; always use a list of column names. Avoid ambiguous chained assignments; when modifying filtered data, use `.copy()` or `.loc` deliberately.

**Predict first:** How many DEMO records are PRODUCING?

In [ ]:
print('One column (Series):')
print(demo['oil_bpd'])
print('Two columns (DataFrame):')
print(demo[['well_id', 'oil_bpd']])
print('Row by label:',demo.loc[1, ['well_id','pressure_psi']].to_dict())
print('First two rows by position:')
print(demo.iloc[:2])

In [ ]:
producing = demo.loc[demo['status'] == 'PRODUCING'].copy()
north_producing = demo.loc[(demo['field'] == 'North') & (demo['status'] == 'PRODUCING')].copy()
print('Producing rows:',len(producing))
print('North + producing:',north_producing[['well_id','oil_bpd']])

### Checkpoint questions
1. What is the difference between `loc` and `iloc`?
2. Why do we write `df[["well_id", "oil_bpd"]]` rather than `df["well_id", "oil_bpd"]`?
3. Why might `.copy()` be useful when producing a filtered table that you will modify later?

# E — Calculated columns, missing values and simple validation (35 minutes)

### Concept introduction
A DataFrame can hold derived operational indicators alongside raw readings. When oil and water volumes are in **barrels per day**, total liquid production is `oil_bpd + water_bpd`. Pressure in psi can be converted to kilopascals by multiplying by `6.8947572932`. Pandas performs arithmetic **element by element** across Series, analogous to NumPy vectorization.

New columns should have informative names and units. A crucial rule is to **preserve missingness**. If oil volume is missing, total liquid volume should ordinarily remain missing rather than silently taking the known water volume as the full total.

`isna()` identifies missing values; `notna()` identifies available values. Do not over-clean this week's source data: missing-data repair strategies are addressed more deeply in the subsequent cleaning topics.

**Common mistakes:** writing `df["oil"] + df["water"]` when the actual column names differ; mutating original teaching data unintentionally; assigning to a filtered view. Use an explicit copied DataFrame.

**Before running:** Predict the first row's total liquid and converted pressure.

In [ ]:
demo_enriched = demo.copy()
demo_enriched['total_liquid_bpd'] = demo_enriched['oil_bpd'] + demo_enriched['water_bpd']
demo_enriched['pressure_kpa'] = demo_enriched['pressure_psi'] * 6.8947572932
print(demo_enriched[['well_id', 'total_liquid_bpd', 'pressure_kpa']].round(2))
print('Original columns still:',len(demo.columns))

In [ ]:
missing_demo = imported.copy()
missing_demo['pressure_kpa'] = missing_demo['pressure_psi'] * 6.8947572932
print(missing_demo[['well_id','pressure_psi','pressure_kpa']])
print('Missing converted pressure:',int(missing_demo['pressure_kpa'].isna().sum()))

# F — Wrap-up, independent lab, and GitHub handoff (20 minutes)

### Bring the ideas together
You should now be able to explain the roles of Series and DataFrames, inspect unknown tables, ingest CSV with dates, select data safely, create numeric features and report missingness. In the separate lab, apply this process to a larger **16-row, four-well synthetic dataset** that is not used in this notebook.

**Lab deliverable:** Complete `pandas_production_lab.py`; create the enriched output; report shape, producing row count and missing values; document your findings. The notebook is the classroom learning resource and **must not replace the assessed `.py` script**.

**GitHub:** Work in your individual repository on `feature/class-02-pandas`. Commit the final script, allowed data, README and any tests; push and open a Pull Request into that same repository's `main`. Do not upload instructor solutions, virtual environments or credentials.

### Exit ticket (answer aloud or in notes)
1. Explain Series versus DataFrame using an oil production example.
2. What happens when a numeric CSV field is blank?
3. Show one Boolean filter.
4. Why should you inspect types and missingness before interpreting results?